# core

> Fill in a module description here

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import yaml, re, sys


In [ ]:
#| export
from dataclasses import dataclass, field

In [ ]:
from fastcore.basics import patch


In [ ]:
#| export
@dataclass
class Styles:
    style: str;
    match_identifiers: list[str] = field(default_factory=set);


In [ ]:
#| export
def load_yaml(path: str) -> dict[str, str]:
    contents = None
    with open(path) as f: contents = f.read()
    return yaml.safe_load(contents)





In [ ]:
#| export
def parse_line(pattern: str, line: str) -> dict[str, str]:
    if pattern is None: return {"_line": line}
    m = re.match(pattern, line)
    if m == None:  return {"_line": f"unparseable line::{line}"}
    return m.groupdict()

In [ ]:
#| export

@dataclass
class Config:
    timestamp_field: str;
    concurrent: bool;
    default_style: str;
    match_identifier_field: str;

    pattern: str | None = None;
    format_str: str | None = None;
    source_field: str | None = None;
    styles: list[Styles] = field(default_factory=list);

    def __post_init__(self):
        self.match_mode = "exact" if self.pattern != None else "contains"
        self.match_identifier_field = "_line" if self.pattern == None else self.match_identifier_field
        if self.pattern and self.match_identifier_field not in re.compile(self.pattern).groupindex: raise ValueError("invalid config: match identifier not contained in pattern regex group")
        if self.pattern and not self.format_str: raise ValueError("invalid config: pattern and format_str are needed")

In [ ]:
#| export
from fastcore.basics import patch

@patch(cls_method=True)
def from_yaml(cls: Config, path: str):
    cnf_dict = load_yaml(path)

    main = cnf_dict.get("parse")
    styles = cnf_dict.get("styles")
    rules = styles.get("rules", [])

    res = Config(
        pattern = main.get("pattern", None),
        format_str= main.get("format_str", None),
        source_field=main.get("source_field", ""),
        timestamp_field= main.get("timestamp_field"),
        concurrent = main.get("concurrent", False),
        default_style=styles.get("default_style", "black"),

        match_identifier_field= styles.get("match_field"),
        styles = list(map(lambda d: Styles(match_identifiers=set(d.get("values", [])), style=d.get("style")), rules))

    )




    return res

In [ ]:
#| export
@patch
def __repr__(self: Config):
    return (f"Config(\n"
            f"  pattern: {self.pattern}\n"
            f"  timestamp_field: {self.timestamp_field}\n"
            f"  source_field: {self.source_field}\n"
            f"  match_field: {self.match_identifier_field}\n"
            f"  styles: {len(self.styles)} rules\n"
            f")")

In [ ]:
#| export
def build_display_string(line_metadata:dict[str, str], format: str ) -> str:
    return format.format_map(line_metadata)

In [ ]:
#| export
def get_styles(cfg: Config, line_metadata: dict[str, str]) -> str:
    match_identifier_value = line_metadata[cfg.match_identifier_field] if "_line" not in line_metadata else "_line"
    if cfg.match_mode == "exact": matching_style: Styles = next((s for s in cfg.styles if match_identifier_value in s.match_identifiers), None)
    else:  matching_style: Styles = next((s for s in cfg.styles if any(ident in line_metadata["_line"] for ident in s.match_identifiers)), None)

    return matching_style.style if matching_style else cfg.default_style

In [ ]:
#| export
def display_line(cfg: Config, line_metadata: dict[str, str])-> None:
    matching_style = get_styles(cfg, line_metadata)
    text = line_metadata.get("_line", line) if cfg.match_mode == "contains" or "_line"  in line_metadata else build_display_string(line_metadata, cfg.format_str)
    console.print(text, style=matching_style, highlight=False)

    return




In [ ]:
#| export
def read_lines(file_name: str | None = None):
    if not file_name and not sys.stdin.isatty():
        for line in sys.stdin: yield line.strip()
    elif file_name:
        with open(file_name) as f:
            for line in f: yield line.strip()
    else:
        raise Error("failure: unable to read input")

In [ ]:
#| export
from rich.console import Console
console = Console()

In [ ]:
#| export
@dataclass
class ConcurrentInfo:
  source: str;
  display_text: str;
  ts: int;
  style: list[str] = field(default_factory=list);


In [ ]:
#| export
def generate_row(n_cols: int,  source_col: int, log: ConcurrentInfo) -> list[str]:
    row = ["" for _ in range(n_cols)]
    row[source_col] = Text(log.display_text, style=log.style)

    return row

In [ ]:
#| export
from rich.text import Text
from rich.table import Table

from fastcore.script import call_parse


def sync_display(cfg: Config, l: str) -> None:
    line_metadata = parse_line(cfg.pattern, l)
    display_line(cfg, line_metadata)


@dataclass
class ConcurrentInfo:
  source: str;
  display_text: str;
  ts: int;
  style: list[str] = field(default_factory=list);


@call_parse
def main(config:str=None,  # Path to config YAML file
         logfile:str=None,
         concurrent:bool=False,
         ): # Path to log file (or pipe via stdin)
          "Pretty print styled logs"
          cfg = Config.from_yaml(config)
          if not cfg.concurrent and not concurrent: 
            for l in read_lines(logfile): sync_display(cfg, l)
          else:
            processes = dict()
            current_idx = 0
            logs = []
            for l in read_lines(logfile):
                line_metadata = parse_line(cfg.pattern, l)
                source = line_metadata[cfg.source_field]
                if source not in processes:
                    processes[source] = current_idx
                    current_idx += 1 

                logs.append(
                  ConcurrentInfo(
                    source=line_metadata.get(cfg.source_field),
                    style=get_styles(cfg, line_metadata),
                    ts=line_metadata.get(cfg.timestamp_field),
                    display_text=build_display_string(line_metadata, cfg.format_str)))

            t = Table()
            t.add_column("time")
            for k in processes: t.add_column(k)

            for log in sorted(logs, key=lambda l: l.ts):
                row = generate_row(len(processes), processes[log.source], log)
                t.add_row(log.ts, *row)

            console.print(t)



In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

In [ ]:
!cat ../loghound/core.py

"""Fill in a module description here"""

# AUTOGENERATED! DO NOT EDIT! File to edit: ../nbs/00_core.ipynb.

# %% auto #0
__all__ = ['console', 'Styles', 'load_yaml', 'parse_line', 'Config', 'build_display_string', 'get_styles', 'display_line',
           'read_lines', 'ConcurrentInfo', 'generate_row', 'sync_display', 'main']

# %% ../nbs/00_core.ipynb #915f694b
import yaml, re, sys


# %% ../nbs/00_core.ipynb #9a7a29b0
from dataclasses import dataclass, field

# %% ../nbs/00_core.ipynb #299e5a84
@dataclass
class Styles:
    style: str;
    match_identifiers: list[str] = field(default_factory=set);


# %% ../nbs/00_core.ipynb #a9c0af2a
def load_yaml(path: str) -> dict[str, str]:
    contents = None
    with open(path) as f: contents = f.read()
    return yaml.safe_load(contents)





# %% ../nbs/00_core.ipynb #db902471
def parse_line(pattern: str, line: str) -> dict[str, str]:
    if pattern is None: return {"_line": line}
    m = re.match(pattern, line)
    if m == None:  return {"_lin

How can I test this nbdev package cli?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

There are a couple of approaches:

1. **Call `main` directly in the notebook** — `@call_parse` lets you call it with keyword args like a normal function:

```python
main(config="path/to/your/config.yaml", logfile="path/to/some.log")
```

2. **As a CLI command** — if you've set up `console_scripts` in your `settings.ini`, you can run it from the terminal after `pip install -e .`. Check your `settings.ini` — do you have a `console_scripts` line pointing to `bloodhound.core:main`?

Do you have a sample YAML config and log file you can test with? That would be the first thing to get in place if not.

In [ ]:
main(config="../example_configs/raft_errors.yaml", logfile="../example_logs/raft.txt", concurrent=True)

┏━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ time   ┃ S0                               ┃ S1                               ┃ S2                               ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 000000 │ 000000::S0::Starting up          │                                  │                                  │
│ 000000 │                                  │ 000000::S1::Starting up          │                                  │
│ 000000 │                                  │                                  │ 000000::S2::Starting up          │
│ 000001 │ 000001::S0::Election timer       │                                  │                                  │
│        │ started                          │                                  │                                  │
│ 000001 │                                  │ 000001::S1::Election timer       │                                  │
│        │                                  │ started                          │                                  │
│ 000001 │                                  │                                  │ 000001::S2::Election timer       │
│        │                                  │                                  │ started                          │
│ 000045 │                                  │                                  │ 000045::S2::Election timeout     │
│        │                                  │                                  │ elapsed                          │
│ 000046 │                                  │                                  │ 000046::S2::Becoming candidate   │
│        │                                  │                                  │ for term 1                       │
│ 000047 │                                  │                                  │ 000047::S2::Requesting vote from │
│        │                                  │                                  │ S0                               │
│ 000047 │                                  │                                  │ 000047::S2::Requesting vote from │
│        │                                  │                                  │ S1                               │
│ 000052 │ 000052::S0::Granting vote to S2  │                                  │                                  │
│        │ for term 1                       │                                  │                                  │
│ 000053 │                                  │ 000053::S1::Granting vote to S2  │                                  │
│        │                                  │ for term 1                       │                                  │
│ 000054 │                                  │                                  │ 000054::S2::Became leader for    │
│        │                                  │                                  │ term 1                           │
│ 000055 │                                  │                                  │ 000055::S2::Leader, checking     │
│        │                                  │                                  │ heartbeats                       │
│ 000056 │                                  │                                  │ 000056::S2::Sending              │
│        │                                  │                                  │ AppendEntries to S0              │
│ 000056 │                                  │                                  │ 000056::S2::Sending              │
│        │                                  │                                  │ AppendEntries to S1              │
│ 000060 │ 000060::S0::Received             │                                  │                                  │
│        │ AppendEntries from S2            │                                  │                                  │
│ 000060 │                                  │ 000060::S1

In order to include the 3rd party packages for nbdev, what must I do?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

In your `settings.ini` file (at the root of your nbdev project), add the third-party packages to the `requirements` field. For your project, that would look something like:

```ini
requirements = pyyaml rich fastcore
```

Each package name is space-separated. This ensures that when someone installs your package (e.g. via `pip install`), those dependencies get pulled in automatically.

Do you have a `settings.ini` already? If so, can you show it so we can check what's there?